# Deckard DVCLive: Native Runtime Walkthrough

This notebook demonstrates Deckard-native DVCLive behavior using a real default example experiment from `examples/sklearn/config/default.yaml`.

No mock experiment objects are used in this flow.

In [1]:
from __future__ import annotations

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from deckard.artifacts import ScoreDict
from deckard.experiment.dvc import run_dvc_experiment_plugin_hook


def ensure_dvclive_available() -> None:
    if importlib.util.find_spec("dvclive") is not None:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dvclive"])


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "deckard").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing pyproject.toml")


ensure_dvclive_available()

REPO_ROOT = find_repo_root(Path.cwd())
DVCLIVE_DIR = REPO_ROOT / "docs/notebooks/build/dvclive_native"
SCORE_ROOT = REPO_ROOT / "examples/sklearn/outputs/logs"

DVCLIVE_DIR

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PosixPath('/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclive_native')

## Build A Real Default Experiment

Compose the default sklearn experiment config and instantiate a real `ExperimentConfig` object.

In [2]:
from deckard.experiment.base import ExperimentConfig


def build_default_example_experiment(repo_root: Path):
    config_dir = repo_root / "examples/sklearn/config"
    GlobalHydra.instance().clear()
    with initialize_config_dir(version_base=None, config_dir=config_dir.as_posix()):
        cfg = compose(config_name="default")

    payload = OmegaConf.to_container(cfg, resolve=True)
    if not isinstance(payload, dict):
        raise TypeError("Expected dict-like Hydra payload for experiment config")

    allowed_keys = set(ExperimentConfig.__dataclass_fields__.keys()) | {"_target_"}
    filtered = {key: value for key, value in payload.items() if key in allowed_keys}
    filtered.setdefault("_target_", "deckard.ExperimentConfig")

    experiment = instantiate(filtered)
    experiment.experiment_name = f"{getattr(experiment, 'experiment_name', 'default')}-dvclive-native"
    return experiment, cfg


experiment, default_cfg = build_default_example_experiment(REPO_ROOT)
type(experiment).__name__, experiment.experiment_name

('ExperimentConfig', 'f117485ab41a0c260f323aaa69e60d61-dvclive-native')

## Configure Native DVCLive Plugin Settings

Enable Deckard DVC hooks and point outputs to a notebook-local build directory.

In [7]:
if DVCLIVE_DIR.exists():
    shutil.rmtree(DVCLIVE_DIR)
DVCLIVE_DIR.mkdir(parents=True, exist_ok=True)

plugin_cfg = {
    "enabled": True,
    "dvclive_dir": DVCLIVE_DIR.as_posix(),
    "mode": "single",
    "pull_dependencies": False,
    "push_outputs": False,
    "make_summary": True,
    "make_report": True,
    "make_dvcyaml": False,
    "report_mode": "html",
    "resume": False,
    "save_dvc_exp": False,
    "cache_images": False,
    "monitor_system": True,
    "fail_on_dvc_error": False,
}

experiment.dvc_plugin = plugin_cfg
experiment()


{'test': {'accuracy': 0.8611,
  'precision': 0.8564,
  'recall': 0.8611,
  'f1': 0.8578,
  'roc_auc': 0.7849,
  'log_loss': 5.0068},
 'data_load_time': 0.08674699999999991,
 'data_sample_time': 0.0012590000000001211,
 'data_pipeline_time': 8.900000000000574e-05,
 'data_score_time': 2.365552,
 'pipeline_fit_time': None,
 'pipeline_transform_time': None,
 'pipeline_y_fit_time': None,
 'pipeline_y_transform_time': None,
 'train_n': 39073,
 'test_n': 9769,
 'files': {},
 'enabled': True,
 'position': 'first',
 'component': 'attack',
 'stage': 'attack-score',
 'event': 'after',
 'executed': False,
 'accuracy': 0.8611,
 'precision': 0.8564,
 'recall': 0.8611,
 'f1': 0.8578,
 'roc_auc': 0.7849,
 'log_loss': 5.0068,
 'training_time': 2.1860999999999997,
 'training_n': 39073,
 'defense_application_time': 0.00011049999739043415,
 'prediction_time': 0.24635300000000093,
 'prediction_n': 9769,
 'prediction_score_time': 0.005807000000000784,
 'benign_accuracy': 0.8,
 'benign_precision': 0.64,
 'ben

In [ ]:
p